<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/lab-05-defect-detector-for-cobalt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5 (graded) — Defect detector for Cobalt
**Course 1: Hands-On Deep Learning with Python — Chapter 5: Convolutional networks for vision**

**Problem brief (Leo Farkas, Cobalt Manufacturing):** "Human inspectors miss ~3% of
defective castings. Build a vision model that flags defective parts on the line."
Target: recall ≥ 0.98 on defects at an acceptable precision, decision in < 200 ms on CPU.

**Dataset:** Kaggle "Casting Product Image Data for Quality Inspection" (requires a free
Kaggle account + API token for `kagglehub`; the offline fallback below generates a small
synthetic image set with the same two-class shape so the notebook runs either way).

**What you'll submit:** a trained CNN (from scratch, then fine-tuned from a `timm`
backbone), a confusion matrix, a Grad-CAM gallery, a CPU latency measurement, and the
recall/precision trade-off discussion.

In [ ]:
!pip install -q timm kagglehub grad-cam

## 1. Load the data (with offline fallback)

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, random_split

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_SIZE = 64  # small on purpose for a free-tier GPU/CPU budget

class CastingDataset(Dataset):
    """Loads real images if kagglehub succeeds; otherwise generates a synthetic stand-in
    with a similar two-class shape (a faint ring artifact marks 'defective') so the rest of
    the notebook — training, eval, Grad-CAM — works unchanged either way."""
    def __init__(self, n_synthetic=1200, img_size=IMG_SIZE):
        self.img_size = img_size
        self.samples = None
        try:
            import kagglehub
            from PIL import Image
            import glob, os
            path = kagglehub.dataset_download('ravirajsinh45/real-life-industrial-dataset-of-casting-product')
            files_ok = glob.glob(os.path.join(path, '**', 'ok_front', '*.jpeg'), recursive=True)
            files_def = glob.glob(os.path.join(path, '**', 'def_front', '*.jpeg'), recursive=True)
            if files_ok and files_def:
                self.samples = [(f, 0) for f in files_ok] + [(f, 1) for f in files_def]
                self.mode = 'real'
                print(f'Loaded the real Kaggle casting dataset: {len(files_ok)} ok, {len(files_def)} defective.')
        except Exception as e:
            print(f'Offline fallback engaged ({e}).')
        if self.samples is None:
            self.mode = 'synthetic'
            rng = np.random.default_rng(0)
            self.synthetic = []
            for i in range(n_synthetic):
                label = int(i % 2 == 0)  # balanced for pipeline development
                base = rng.normal(0.5, 0.08, (img_size, img_size)).astype(np.float32)
                if label == 1:
                    yy, xx = np.mgrid[0:img_size, 0:img_size]
                    cy, cx = rng.integers(20, img_size - 20, 2)
                    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
                    ring = np.exp(-((r - 10) ** 2) / 8.0) * 0.5
                    base = np.clip(base + ring, 0, 1)
                self.synthetic.append((base, label))
            print(f'Synthetic set: {n_synthetic} images, balanced ok/defective.')

    def __len__(self):
        return len(self.samples) if self.mode == 'real' else len(self.synthetic)

    def __getitem__(self, i):
        if self.mode == 'real':
            from PIL import Image
            path, label = self.samples[i]
            img = Image.open(path).convert('L').resize((self.img_size, self.img_size))
            arr = np.array(img, dtype=np.float32) / 255.0
        else:
            arr, label = self.synthetic[i]
        return torch.tensor(arr).unsqueeze(0), torch.tensor(label, dtype=torch.long)


full_ds = CastingDataset()
n_val = int(len(full_ds) * 0.2)
train_ds, val_ds = random_split(full_ds, [len(full_ds) - n_val, n_val], generator=torch.Generator().manual_seed(0))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)
print(f'train={len(train_ds)}  val={len(val_ds)}')

## 2. A small CNN from scratch
Fill in the `TODO`s: a conv block (conv → ReLU → pool), stacked twice, then a classifier head.

In [ ]:
import torch.nn as nn

class _NotImplementedBlock(nn.Module):
    """Placeholder so an unfilled TODO fails clearly and immediately, instead of silently
    passing the input through unchanged and crashing later with a confusing shape-mismatch
    error deep in the classifier head."""
    def forward(self, x):
        raise NotImplementedError('Fill in the two conv blocks in SmallCNN.features above.')


class SmallCNN(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        # TODO: two conv blocks, e.g.
        # nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        # nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        self.features = _NotImplementedBlock()  # replace with nn.Sequential(...) per the TODO above
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, n_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def train_model(model, loader, val_loader, n_epochs=10, lr=1e-3, class_weights=None):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    weight = torch.tensor(class_weights, dtype=torch.float32).to(device) if class_weights else None
    loss_fn = nn.CrossEntropyLoss(weight=weight)
    for epoch in range(n_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        if epoch % 3 == 0 or epoch == n_epochs - 1:
            print(f'epoch {epoch}: loss={loss.item():.4f}')
    return model

cnn = SmallCNN()
cnn = train_model(cnn, train_loader, val_loader, n_epochs=10, class_weights=[1.0, 1.0])

## 3. Evaluate: confusion matrix, recall/precision

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb.to(device))
            preds.extend(out.argmax(1).cpu().numpy())
            trues.extend(yb.numpy())
    return np.array(trues), np.array(preds)

y_true, y_pred = evaluate(cnn, val_loader)
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=['ok', 'defective']))
print('Target: recall for the defective class >= 0.98. If you are not there yet, revisit class')
print('weighting, augmentation, or the transfer-learning model below.')

## 4. Transfer learning from a `timm` backbone

In [ ]:
import timm

class TimmBackboneClassifier(nn.Module):
    def __init__(self, n_classes=2):
        super().__init__()
        self.backbone = timm.create_model('resnet18', pretrained=True, num_classes=n_classes, in_chans=1)

    def forward(self, x):
        return self.backbone(x)

transfer_model = TimmBackboneClassifier()
transfer_model = train_model(transfer_model, train_loader, val_loader, n_epochs=6, lr=1e-4, class_weights=[1.0, 2.0])
y_true_t, y_pred_t = evaluate(transfer_model, val_loader)
print(classification_report(y_true_t, y_pred_t, target_names=['ok', 'defective']))

## 5. Grad-CAM gallery: why did it flag this part?

In [ ]:
from pytorch_grad_cam import GradCAM
import matplotlib.pyplot as plt

target_layer = [transfer_model.backbone.layer4[-1]]
cam = GradCAM(model=transfer_model, target_layers=target_layer)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    xb, yb = val_ds[i]
    grayscale_cam = cam(input_tensor=xb.unsqueeze(0).to(device))[0]
    axes[i].imshow(xb.squeeze(), cmap='gray')
    axes[i].imshow(grayscale_cam, cmap='jet', alpha=0.5)
    axes[i].set_title(f"true={'defective' if yb == 1 else 'ok'}")
    axes[i].axis('off')
plt.suptitle('Grad-CAM: red = drove the prediction')
plt.show()

## 6. CPU latency measurement (must be < 200 ms)

In [ ]:
import time

cpu_model = transfer_model.to('cpu').eval()
x_sample = val_ds[0][0].unsqueeze(0)

with torch.no_grad():
    cpu_model(x_sample)  # warm-up
    t0 = time.perf_counter()
    for _ in range(20):
        cpu_model(x_sample)
    elapsed_ms = (time.perf_counter() - t0) / 20 * 1000

print(f'Mean CPU latency per image: {elapsed_ms:.1f} ms  (budget: 200 ms)')
transfer_model.to(device)  # move back for any further GPU work

## 7. Error analysis + decision-dossier section (fill in)
Look at a few false negatives (missed defects) and false positives (false alarms) from the
confusion matrix above. What do they have in common? State the recall/precision trade-off
you're recommending to Cobalt, and why a missed defect and a false alarm are not equally
costly.

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 5: Convolutional networks for vision*